In [5]:
# Cell 1 — Imports
# -------------------------------------------------------------------
import os
import glob
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go


In [6]:
# 📌 Cell 2 (REPLACE) — Locate latest data_04 + data_07 inputs (07=thread_panel_stance.csv only)
# -------------------------------------------------------------------
import os
import glob
from pathlib import Path

try:
    from pyprojroot import here
    PROJECT_ROOT = here()
except Exception:
    PROJECT_ROOT = Path(os.getcwd()).resolve().parent

DATA_ROOT = Path(PROJECT_ROOT) / "Data"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)

def pick_latest_dir(pattern: str) -> Path:
    cands = [Path(p) for p in glob.glob(str(DATA_ROOT / pattern)) if Path(p).is_dir()]
    if not cands:
        return Path("")
    return max(cands, key=lambda p: p.stat().st_mtime)

# ✅ 04：情绪（母评论 valence）
DATA_04_DIR = pick_latest_dir("data_04_sentiment5*")
if not DATA_04_DIR:
    raise FileNotFoundError("❌ 找不到 Data/data_04_sentiment5*，请先跑 04_sentiment_analysis.ipynb")

TOP_PARQUET = DATA_04_DIR / "sentiment5_top_level.parquet"
REPLY_PARQUET = DATA_04_DIR / "sentiment5_replies.parquet"

if not TOP_PARQUET.exists():
    raise FileNotFoundError(f"❌ 找不到情绪 top-level parquet: {TOP_PARQUET}")

# ✅ 07：thread stance（你确认只有这个文件）
DATA_07_DIR = pick_latest_dir("data_07_thread_stance_nli*")
if not DATA_07_DIR:
    raise FileNotFoundError("❌ 找不到 Data/data_07_thread_stance_nli*，请先跑 07_thread_stance_nli.ipynb")

THREAD_PANEL_STANCE = DATA_07_DIR / "thread_panel_stance.csv"
if not THREAD_PANEL_STANCE.exists():
    raise FileNotFoundError(f"❌ 找不到 07 的 thread_panel_stance.csv: {THREAD_PANEL_STANCE}")

print("✅ DATA_04_DIR:", DATA_04_DIR)
print("✅ TOP_PARQUET:", TOP_PARQUET)
print("✅ REPLY_PARQUET exists:", REPLY_PARQUET.exists(), REPLY_PARQUET)

print("✅ DATA_07_DIR:", DATA_07_DIR)
print("✅ THREAD_PANEL_STANCE:", THREAD_PANEL_STANCE)


PROJECT_ROOT = d:\coding\projects\Python\Youtube Comment Sentiment Analysis
DATA_ROOT = d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data
✅ DATA_04_DIR: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_04_sentiment5_20260227_181316
✅ TOP_PARQUET: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_04_sentiment5_20260227_181316\sentiment5_top_level.parquet
✅ REPLY_PARQUET exists: False d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_04_sentiment5_20260227_181316\sentiment5_replies.parquet
✅ DATA_07_DIR: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211
✅ THREAD_PANEL_STANCE: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_07_thread_stance_nli_20260301_010211\thread_panel_stance.csv


In [7]:
# Cell 3 (REPLACE) — Create 08 output folder + fig helpers
# -------------------------------------------------------------------
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(DATA_ROOT, f"data_08_topic_modeling_{timestamp}")
FIG_DIR = os.path.join(RUN_DIR, "figs")

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

def out(name: str) -> str:
    return os.path.join(RUN_DIR, name)

def figout(name: str) -> str:
    return os.path.join(FIG_DIR, name)

def save_plotly(fig, path):
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    print("✅ Saved:", path)

print("✅ RUN_DIR:", RUN_DIR)
print("✅ FIG_DIR:", FIG_DIR)


✅ RUN_DIR: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952
✅ FIG_DIR: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952\figs


In [10]:
# Cell 4 (REPLACE) — Load sentiment parents (04) + merge engagement from 07 thread_panel_stance.csv (robust coalesce)
# -------------------------------------------------------------------
import pandas as pd
import numpy as np

top = pd.read_parquet(TOP_PARQUET)

# unify text
if "text_model" in top.columns:
    top = top.rename(columns={"text_model": "text"})
elif "text_clean" in top.columns:
    top = top.rename(columns={"text_clean": "text"})

need = ["thread_id", "cell_id", "text"]
for c in need:
    if c not in top.columns:
        raise KeyError(f"❌ sentiment parquet missing {c}. got: {list(top.columns)[:30]} ...")

if "sent5_valence_expected" not in top.columns:
    raise KeyError("❌ sentiment parquet missing sent5_valence_expected（请确认 04 输出是否正确）")

df = top.dropna(subset=["thread_id", "cell_id", "text"]).copy()
df["thread_id"] = df["thread_id"].astype(str)
df["text"] = df["text"].astype(str).str.strip()

# filters
df["token_count"] = df["text"].str.split().str.len()
df = df[df["token_count"] >= 5].copy()
df = df[df["cell_id"].str.contains("labor", na=False, case=False)].copy()
df = df.drop_duplicates("thread_id").copy()

print("✅ df base(04 sentiment):", df.shape)

# --- merge engagement from 07 thread_panel_stance.csv ---
panel07 = pd.read_csv(THREAD_PANEL_STANCE)
if "thread_id" not in panel07.columns:
    raise KeyError(f"❌ 07 thread_panel_stance.csv missing thread_id: {THREAD_PANEL_STANCE}")

panel07["thread_id"] = panel07["thread_id"].astype(str)

# detect engagement cols inside 07 file
like_col = next((c for c in ["like_count_parent","like_count","parent_like_count","likes_parent","likes"] if c in panel07.columns), None)
reply_col = next((c for c in ["total_reply_count","reply_count","parent_reply_count","replies_parent","replies"] if c in panel07.columns), None)

print("✅ 07 detected like_col:", like_col)
print("✅ 07 detected reply_col:", reply_col)

keep = ["thread_id"]
if like_col: keep.append(like_col)
if reply_col: keep.append(reply_col)

if len(keep) == 1:
    print("⚠️ 07 thread_panel_stance.csv does NOT contain likes/replies columns. amplification may be 0 unless you merge from another source.")
else:
    panel07 = panel07[keep].drop_duplicates("thread_id").copy()

    # normalize names in panel07 before merge
    rename_map = {}
    if like_col and like_col != "like_count_parent":
        rename_map[like_col] = "like_count_parent"
    if reply_col and reply_col != "total_reply_count":
        rename_map[reply_col] = "total_reply_count"
    panel07 = panel07.rename(columns=rename_map)

    # 如果 df 里已经有同名列（哪怕是空的），merge 会产生 _x/_y
    df = df.merge(panel07, on="thread_id", how="left", suffixes=("_x", "_y"))
    print("✅ merged engagement from 07:", panel07.shape)

    # coalesce helper: pick y first, else x, else existing
    def coalesce_cols(df_, base):
        a = f"{base}_y"
        b = f"{base}_x"
        if a in df_.columns and b in df_.columns:
            df_[base] = df_[a].combine_first(df_[b])
            df_.drop(columns=[a, b], inplace=True)
        elif a in df_.columns and base not in df_.columns:
            df_.rename(columns={a: base}, inplace=True)
        elif b in df_.columns and base not in df_.columns:
            df_.rename(columns={b: base}, inplace=True)
        return df_

    df = coalesce_cols(df, "like_count_parent")
    df = coalesce_cols(df, "total_reply_count")

# final checks
print("✅ has like_count_parent?", "like_count_parent" in df.columns,
      "| non-null:", df["like_count_parent"].notna().mean() if "like_count_parent" in df.columns else None)
print("✅ has total_reply_count?", "total_reply_count" in df.columns,
      "| non-null:", df["total_reply_count"].notna().mean() if "total_reply_count" in df.columns else None)


✅ df base(04 sentiment): (22520, 26)
✅ 07 detected like_col: like_count_parent
✅ 07 detected reply_col: total_reply_count
✅ merged engagement from 07: (22520, 3)
✅ has like_count_parent? True | non-null: 1.0
✅ has total_reply_count? True | non-null: 1.0


In [11]:
# Cell 5 (REPLACE) — Detect columns (valence/likes/replies) after 07 merge
# -------------------------------------------------------------------
import pandas as pd

df["sent5_valence_expected"] = pd.to_numeric(df["sent5_valence_expected"], errors="coerce")
df["like_count_parent"] = pd.to_numeric(df["like_count_parent"], errors="coerce")
df["total_reply_count"] = pd.to_numeric(df["total_reply_count"], errors="coerce")

valence_col = "sent5_valence_expected"
likes_col = "like_count_parent"
replies_col = "total_reply_count"

print("✅ valence non-null:", df[valence_col].notna().mean(), "| mean:", df[valence_col].mean())
print("✅ likes non-null:", df[likes_col].notna().mean(), "| mean:", df[likes_col].mean())
print("✅ replies non-null:", df[replies_col].notna().mean(), "| mean:", df[replies_col].mean())


✅ valence non-null: 1.0 | mean: -0.05617849866922333
✅ likes non-null: 1.0 | mean: 62.125
✅ replies non-null: 1.0 | mean: 1.3848134991119005


In [12]:
# Cell 6 — Compute embeddings (Local Model)
# -------------------------------------------------------------------
import os
import numpy as np
from sentence_transformers import SentenceTransformer

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
EMB_MODEL_PATH = os.path.join(MODELS_DIR, "embedding_all-MiniLM-L6-v2")

# 🌟 换一个新的缓存名，适配过滤后的干净数据
EMB_CACHE = out("embeddings_minilm_labor_tokens5.npy")

texts = df["text"].tolist()

if os.path.exists(EMB_CACHE):
    print(f"⏳ 发现本地缓存，正在加载: {EMB_CACHE} ...")
    emb = np.load(EMB_CACHE)
else:
    print(f"⏳ 正在编码 {len(texts)} 条高质量文本 (可能需要几分钟)...")
    model = SentenceTransformer(EMB_MODEL_PATH)
    emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMB_CACHE, emb)
    
print("✅ 向量化资产准备完毕！维度:", emb.shape)

⏳ 正在编码 22520 条高质量文本 (可能需要几分钟)...


Batches:   0%|          | 0/352 [00:00<?, ?it/s]

✅ 向量化资产准备完毕！维度: (22520, 384)


In [13]:
# Cell 7 (REPLACE) — PCA + KMeans clustering (stable)
# -------------------------------------------------------------------
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 你原 cell 6 已经算 emb（normalize_embeddings=True）
pca = PCA(n_components=100, random_state=42)
X = pca.fit_transform(emb)

K = 50  # 可调：30更粗，80更细
km = KMeans(n_clusters=K, random_state=42, n_init="auto")
df["topic_id"] = km.fit_predict(X)

print("✅ KMeans done. topics:", df["topic_id"].nunique())
print(df["topic_id"].value_counts().head(10))

✅ KMeans done. topics: 50
topic_id
37    1006
43     824
46     762
25     689
29     651
35     638
5      637
30     636
12     618
11     590
Name: count, dtype: int64


In [14]:
# Cell 8 — c-TF-IDF for Distinctive Keywords (BERTopic 核心数学还原版)
# -------------------------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import numpy as np

# 1. 自定义停用词表
custom_stop_words = list(ENGLISH_STOP_WORDS) + [
    "shein", "zara", "h&m", "just", "like", "im", "dont", "people", 
    "clothes", "video", "watch", "said", "say", "going", "know", "think",
    "really", "got", "didnt", "theyre", "thats", "ive", "things", "way"
]

# 🌟 优化 1：Token Pattern 修复（保留单引号缩写与否定，防止 don't 变 don）
pure_word_pattern = r"(?u)\b[a-zA-Z][a-zA-Z']+\b"

# 剔除 -1 噪声，将同一话题拼成“宏文档”
valid_df = df[df["topic_id"] != -1].copy()
docs_per_topic = valid_df.groupby("topic_id")["text"].apply(lambda x: " ".join(x)).reset_index()

# 提取词频矩阵
cv = CountVectorizer(stop_words=custom_stop_words, ngram_range=(1,2), token_pattern=pure_word_pattern, max_features=30000)
X = cv.fit_transform(docs_per_topic["text"]).toarray() 

# ==========================================================
# 🌟 优化 2：手写纯正的 Class-based TF-IDF
# ==========================================================

# 1. Class-TF (按类文档的“总词数”进行除法归一化，而不是 l2)
# 公式: Term Freq in Class / Total Words in Class
tf = X / (X.sum(axis=1, keepdims=True) + 1e-9)

# 2. Class-IDF (经典的对数惩罚机制)
# 公式: log( 1 + 类别总数 / 包含该词的类别数 )
idf = np.log(1 + (X.shape[0] / (np.sum(X > 0, axis=0) + 1e-9)))

# 3. 得到最纯正的 c-TF-IDF 矩阵
c_tfidf = tf * idf

terms = np.array(cv.get_feature_names_out())
topic_ids_valid = docs_per_topic["topic_id"].tolist()

topic_keywords = {}
for i, t in enumerate(topic_ids_valid):
    v = c_tfidf[i].ravel()
    top_idx = np.argsort(-v)[:12]
    topic_keywords[t] = terms[top_idx].tolist()

topic_keywords[-1] = [] 

print("✅ 纯正 c-TF-IDF 区分性高频词提取完毕！预览前 3 个话题：")
for t in topic_ids_valid[:3]:
    print(f"Topic {t}: {topic_keywords[t][:6]}")

✅ 纯正 c-TF-IDF 区分性高频词提取完毕！预览前 3 个话题：
Topic 0: ['jeans', 'fur', 'barrel jeans', 'wear', 'wearing', 'faux fur']
Topic 1: ['cheap', 'money', 'buy', 'cost', 'price', 'pay']
Topic 2: ['chinese', 'china', 'brands', 'nike', 'products', 'chinese brands']


In [15]:
# Cell 9 — Centroid Representatives & Summarization
# -------------------------------------------------------------------
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

print("⏳ 正在计算向量重心 (Centroids) 以提取 Top 3 代表性评论...")
topic_examples = {}

for t in df["topic_id"].unique():
    if t == -1: 
        continue
        
    # 🌟 优化 2：计算簇中心，提取距离最近的 Top 3 评论
    idx = np.where(df["topic_id"] == t)[0]
    cluster_embs = emb[idx]
    
    centroid = cluster_embs.mean(axis=0).reshape(1, -1)
    sims = cosine_similarity(cluster_embs, centroid).ravel()
    
    top_k = min(3, len(idx))
    top_local_idx = np.argsort(-sims)[:top_k]
    top_global_idx = idx[top_local_idx]
    
    topic_examples[t] = df.iloc[top_global_idx]["text"].tolist()

# ---------------------------------------------------------
# (可选) 喂给大模型生成摘要，用的就是最纯净的 Top 3
# ---------------------------------------------------------
SUM_MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "summarizer_distilbart-cnn-12-6")
summarizer = pipeline("summarization", model=SUM_MODEL_PATH)

topic_summaries = {}
MAX_SUMMARIES = 30
target_topics = df[df["topic_id"] != -1]["topic_id"].value_counts().head(MAX_SUMMARIES).index.tolist()

for t in target_topics:
    # 直接用最纯的代表性评论拼接，杜绝随机性幻觉
    blob = " ".join(topic_examples.get(t, []))[:2800]
    try:
        summary_out = summarizer(blob, max_length=60, min_length=15, do_sample=False, truncation=True)
        topic_summaries[t] = summary_out[0]["summary_text"]
    except Exception as e:
        topic_summaries[t] = ""

print("✅ 代表性样本提取 & 摘要生成完毕！")

⏳ 正在计算向量重心 (Centroids) 以提取 Top 3 代表性评论...


Device set to use cpu
Your max_length is set to 60, but your input_length is only 37. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=18)
Your max_length is set to 60, but your input_length is only 51. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=25)
Your max_length is set to 60, but your input_length is only 39. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)
Your max_length is set to 60, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', m

✅ 代表性样本提取 & 摘要生成完毕！


In [27]:
# Cell 10 (REPLACE) — Topic metrics + stance-aware destructiveness (zscore fusion, improved resonance + renorm)
# -------------------------------------------------------------------
import pandas as pd
import numpy as np

# ---------- 1) base per-row scores ----------
df["_neg"] = (-df[valence_col].astype(float)).clip(lower=0)
df["_amp"] = 0.0
if likes_col:
    df[likes_col] = pd.to_numeric(df[likes_col], errors="coerce")
    df["_amp"] += df[likes_col].fillna(0.0)
if replies_col:
    df[replies_col] = pd.to_numeric(df[replies_col], errors="coerce")
    df["_amp"] += 2.0 * df[replies_col].fillna(0.0)

# ---------- 2) topic basic aggregation ----------
topic_rows = []
for t in sorted(df["topic_id"].unique().tolist()):
    sub = df[df["topic_id"] == t]

    top6_words = topic_keywords.get(t, [])[:6]
    top3_examples = topic_examples.get(t, [])

    row = {
        "topic_id": t,
        "n_comments": int(len(sub)),
        "label": " | ".join(top6_words) if top6_words else str(t),
        "examples": "<br><br>".join([f"🗣️ {ex}" for ex in top3_examples]) if top3_examples else "",
        "keywords_short": ", ".join(top6_words[:4]) if top6_words else "",
        "summary": topic_summaries.get(t, "")
    }

    row["net_negative"] = float(sub["_neg"].mean())
    row["amplification"] = float(sub["_amp"].mean())
    topic_rows.append(row)

topic_summary = pd.DataFrame(topic_rows)

# ---------- 3) stance aggregation from 07 (thread_panel_stance.csv) ----------
for c in [
    "support_rate_wavg","oppose_rate_wavg","neutral_rate_wavg",
    "controversy_wavg","resonance_wavg","avg_reply_weight","threads_with_replies"
]:
    topic_summary[c] = 0.0

if THREAD_PANEL_STANCE.exists():
    tp = pd.read_csv(THREAD_PANEL_STANCE)
    if "thread_id" not in tp.columns:
        raise KeyError(f"❌ 07 thread_panel_stance.csv missing thread_id: {THREAD_PANEL_STANCE}")
    tp["thread_id"] = tp["thread_id"].astype(str)

    # ✅ coalesce *_y/_x into base names (your 07 file may contain suffixes)
    def coalesce_cols(df_, base):
        y = f"{base}_y"
        x = f"{base}_x"
        if base in df_.columns:
            return
        if y in df_.columns and x in df_.columns:
            df_[base] = df_[y].combine_first(df_[x])
        elif y in df_.columns:
            df_[base] = df_[y]
        elif x in df_.columns:
            df_[base] = df_[x]

    for col in ["support_rate","oppose_rate","neutral_rate","controversy","resonance","w_total",
                "support","oppose","neutral"]:
        coalesce_cols(tp, col)

    # ensure required columns exist
    for c in ["support_rate","oppose_rate","neutral_rate","controversy","resonance","w_total"]:
        if c not in tp.columns:
            tp[c] = 0.0

    # thread -> topic mapping
    thread_topic = df[["thread_id","topic_id"]].drop_duplicates("thread_id")
    thread_topic["thread_id"] = thread_topic["thread_id"].astype(str)
    tp = tp.merge(thread_topic, on="thread_id", how="inner")

    tp["_w"] = pd.to_numeric(tp["w_total"], errors="coerce").fillna(0.0)
    tp.loc[tp["_w"] <= 0, "_w"] = 1.0

    def wavg(g, col):
        v = pd.to_numeric(g[col], errors="coerce").fillna(0.0)
        return float((v * g["_w"]).sum() / (g["_w"].sum() + 1e-9))

    rows = []
    for tid, g in tp.groupby("topic_id"):
        rows.append({
            "topic_id": tid,
            "support_rate_wavg": wavg(g,"support_rate"),
            "oppose_rate_wavg":  wavg(g,"oppose_rate"),
            "neutral_rate_wavg": wavg(g,"neutral_rate"),
            "controversy_wavg":  wavg(g,"controversy"),
            "resonance_wavg":    wavg(g,"resonance"),  # 旧字段保留
            "avg_reply_weight":  float(g["_w"].mean()),
            "threads_with_replies": int((pd.to_numeric(g["w_total"], errors="coerce").fillna(0.0) > 0).sum()),
        })
    stance_topic = pd.DataFrame(rows)

    topic_summary = topic_summary.merge(stance_topic, on="topic_id", how="left", suffixes=("","_new"))
    for c in ["support_rate_wavg","oppose_rate_wavg","neutral_rate_wavg","controversy_wavg","resonance_wavg","avg_reply_weight","threads_with_replies"]:
        if f"{c}_new" in topic_summary.columns:
            topic_summary[c] = topic_summary[f"{c}_new"].fillna(topic_summary[c])
            topic_summary.drop(columns=[f"{c}_new"], inplace=True)

# ---------- 4) renormalize stance rates (support+oppose+neutral -> 1) ----------
for c in ["support_rate_wavg","oppose_rate_wavg","neutral_rate_wavg","controversy_wavg","resonance_wavg","avg_reply_weight"]:
    topic_summary[c] = pd.to_numeric(topic_summary[c], errors="coerce").fillna(0.0)

rate_sum = topic_summary["support_rate_wavg"] + topic_summary["oppose_rate_wavg"] + topic_summary["neutral_rate_wavg"]
mask = rate_sum > 1e-9
topic_summary.loc[mask, "support_rate_wavg"] = topic_summary.loc[mask, "support_rate_wavg"] / rate_sum[mask]
topic_summary.loc[mask, "oppose_rate_wavg"]  = topic_summary.loc[mask, "oppose_rate_wavg"]  / rate_sum[mask]
topic_summary.loc[mask, "neutral_rate_wavg"] = topic_summary.loc[mask, "neutral_rate_wavg"] / rate_sum[mask]

# ---------- 5) improved resonance definition (direction + strength) ----------
topic_summary["polarity_wavg"] = (topic_summary["support_rate_wavg"] - topic_summary["oppose_rate_wavg"]).clip(-1, 1)
topic_summary["alignment_wavg"] = (1.0 - topic_summary["neutral_rate_wavg"]).clip(0, 1)
topic_summary["resonance_strength_wavg"] = (topic_summary["alignment_wavg"] * topic_summary["polarity_wavg"].abs()).clip(0, 1)

rate_sum2 = topic_summary["support_rate_wavg"] + topic_summary["oppose_rate_wavg"] + topic_summary["neutral_rate_wavg"]
print("✅ stance rate_sum (after renorm) mean/min/max:", float(rate_sum2.mean()), float(rate_sum2.min()), float(rate_sum2.max()))
print("✅ stance sanity:", topic_summary[["avg_reply_weight","controversy_wavg","resonance_wavg","resonance_strength_wavg"]].describe())

# ---------- 6) destructiveness — zscore fusion (base + stance) ----------
topic_summary["volume_weight"] = np.log1p(topic_summary["n_comments"].astype(float))
topic_summary["risk_penalty"] = topic_summary["net_negative"].clip(lower=0)

def zscore(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    if s.std() == 0 or np.isnan(s.std()):
        return pd.Series([0.0]*len(s), index=s.index)
    return (s - s.mean()) / (s.std() + 1e-9)

topic_summary["destructiveness_base"] = (
    zscore(topic_summary["risk_penalty"]) + zscore(topic_summary["amplification"])
) * topic_summary["volume_weight"]

ALPHA_RESO = 0.7
BETA_CONT = 0.7

topic_summary["destructiveness_stance"] = (
    zscore(topic_summary["risk_penalty"])
    + zscore(topic_summary["amplification"])
    + ALPHA_RESO * zscore(topic_summary["resonance_strength_wavg"])  # ✅ improved resonance
    + BETA_CONT  * zscore(topic_summary["controversy_wavg"])
) * topic_summary["volume_weight"]

topic_summary.loc[topic_summary["net_negative"] <= 0, ["destructiveness_base","destructiveness_stance"]] = 0.0
topic_summary["priority"] = topic_summary["destructiveness_stance"]

topic_summary = topic_summary.sort_values("priority", ascending=False).reset_index(drop=True)

print("✅ topic_summary preview:")
display(topic_summary[[
    "topic_id","label","net_negative","amplification",
    "destructiveness_base","destructiveness_stance","priority",
    "resonance_strength_wavg","controversy_wavg","avg_reply_weight"
]].head(10))


✅ stance rate_sum (after renorm) mean/min/max: 1.0 0.9999999999999998 1.0
✅ stance sanity:        avg_reply_weight  controversy_wavg  resonance_wavg  \
count         50.000000         50.000000       50.000000   
mean          10.166438          0.695182        0.022118   
std            8.211990          0.149694        0.025711   
min            1.168224          0.148000        0.000000   
25%            4.834273          0.620675        0.009534   
50%            7.751062          0.690164        0.013237   
75%           13.619614          0.818432        0.024773   
max           41.412281          0.915074        0.144978   

       resonance_strength_wavg  
count                50.000000  
mean                  0.035689  
std                   0.031834  
min                   0.001318  
25%                   0.012117  
50%                   0.024261  
75%                   0.049766  
max                   0.140787  
✅ topic_summary preview:


,topic_id,label,net_negative,amplification,destructiveness_base,destructiveness_stance,priority,resonance_strength_wavg,controversy_wavg,avg_reply_weight
0,15,emoji | adele | peele | nancy | girl | woman,0.225179,347.531780,23.987782,33.137499,33.137499,0.113418,0.647366,20.709746
1,9,influencers | influencer | influencers influen...,0.493946,205.864583,17.450343,18.220450,18.220450,0.010603,0.844437,14.932292
2,38,fashion | fast fashion | fast | buy | buy fast...,0.346705,179.081871,11.778052,13.732532,13.732532,0.017894,0.850455,41.412281
3,48,buy | emoji | bought | i'm | company | buying,0.555284,91.816162,11.912035,12.607482,12.607482,0.008759,0.845772,19.543434
4,18,slavery | slaves | slave | modern slavery | mo...,0.592976,26.957684,6.976102,9.712549,9.712549,0.065193,0.652234,6.481069
5,21,clothing | fashion | brands | buy | workers | ...,0.501164,75.908367,8.386816,9.651786,9.651786,0.007680,0.870370,16.135458
6,45,companies | company | jaba | business | don't ...,0.572643,52.978142,8.975712,9.275616,9.275616,0.026192,0.750000,8.080146
7,13,wear | fashion | outfit | buy | outfits | trends,0.152203,188.757106,5.959494,7.999274,7.999274,0.024299,0.821916,31.457364
8,47,wear | years | clothing | buy | worn | throw,0.227385,140.510204,4.291115,7.258468,7.258468,0.047905,0.741911,27.403628
9,10,perfumes | perfume | jasmine | fragrance | fra...,0.529000,51.109589,6.103379,6.649264,6.649264,0.024223,0.770740,10.237443


In [29]:
# Cell 11 (REPLACE) — Topic lift across 2×2 cells + merge FULL topic_summary
# -------------------------------------------------------------------
import numpy as np
import pandas as pd

cells = sorted(df["cell_id"].unique().tolist())
print("cells:", cells)

# share(topic, cell) = topic_comments_in_cell / total_comments_in_cell
cell_totals = df.groupby("cell_id").size().rename("cell_total").reset_index()
topic_cell = df.groupby(["topic_id", "cell_id"]).size().rename("n").reset_index()
topic_cell = topic_cell.merge(cell_totals, on="cell_id", how="left")
topic_cell["share"] = topic_cell["n"] / (topic_cell["cell_total"] + 1e-9)

pivot = topic_cell.pivot_table(index="topic_id", columns="cell_id", values="share", fill_value=0.0).reset_index()

for c in ["shein_labor", "non_shein_labor", "shein_environment", "non_shein_environment"]:
    if c not in pivot.columns:
        pivot[c] = 0.0

pivot["lift_sheinLabor_vs_nonSheinLabor"] = pivot["shein_labor"] - pivot["non_shein_labor"]
pivot["ratio_sheinLabor_vs_nonSheinLabor"] = pivot["shein_labor"] / (pivot["non_shein_labor"] + 1e-9)

# ✅ merge FULL topic_summary (so new columns like resonance_strength_wavg exist)
topic_lift = pivot.merge(topic_summary, on="topic_id", how="left")

print("✅ topic_lift merged:", topic_lift.shape)
print("✅ topic_lift columns contains resonance_strength_wavg?",
      "resonance_strength_wavg" in topic_lift.columns)

display(topic_lift.sort_values("lift_sheinLabor_vs_nonSheinLabor", ascending=False).head(3))


cells: ['non_shein_labor', 'shein_labor']
✅ topic_lift merged: (50, 29)
✅ topic_lift columns contains resonance_strength_wavg? True


,topic_id,non_shein_labor,shein_labor,shein_environment,non_shein_environment,lift_sheinLabor_vs_nonSheinLabor,ratio_sheinLabor_vs_nonSheinLabor,n_comments,label,examples,...,avg_reply_weight,threads_with_replies,polarity_wavg,alignment_wavg,resonance_strength_wavg,volume_weight,risk_penalty,destructiveness_base,destructiveness_stance,priority
48,48,0.000264,0.066658,0.0,0.0,0.066394,252.531422,495,buy | emoji | bought | i'm | company | buying,🗣️ never heard of shein but i will happily con...,...,19.543434,69,-0.070934,0.123486,0.008759,6.206576,0.555284,11.912035,12.607482,12.607482
4,4,0.000132,0.056204,0.0,0.0,0.056072,425.855856,416,buy | clothing | it's | fashion | don't | cheap,🗣️ who is still shopping from shein? their clo...,...,14.514423,71,-0.088003,0.116459,0.010249,6.033086,0.504298,5.263038,6.030215,6.030215
29,29,0.013198,0.061227,0.0,0.0,0.048029,4.639189,651,buy | thrift | clothing | fashion | wear | cheap,🗣️ i’m southeast asian and even the cheapest s...,...,19.462366,92,-0.030138,0.094542,0.002849,6.480045,0.257799,0.043400,1.233365,1.233365


In [18]:
# Cell 12.1 — Topic Bubble Chart
import plotly.express as px

def save_plotly(fig, path):
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    print("✅ Saved:", path)

viz = topic_lift[topic_lift["topic_id"] != -1].copy()
viz["net_negative"] = viz["net_negative"].fillna(0)
viz["amplification"] = viz["amplification"].fillna(0)

# 🌟 使用 ID + 短高频词汇作为标签，极致清爽
viz["topic_label"] = viz["topic_id"].astype(str) + " - [" + viz["keywords_short"].fillna("") + "]"

fig1 = px.scatter(
    viz, 
    x="net_negative", 
    y="amplification", 
    size="n_comments", 
    color="lift_sheinLabor_vs_nonSheinLabor",
    color_continuous_scale="RdBu_r", 
    color_continuous_midpoint=0,     
    size_max=50,                     
    hover_name="topic_label", 
    hover_data={"topic_id": False, "summary": True, "n_comments": True, "net_negative": ':.2f', "amplification": ':.1f', "topic_label": False},
    title="Topic Landscape: Negativity × Amplification (Bubble Size = Volume, Color = SHEIN Lift)"
)

fig1.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig1.update_layout(xaxis_title="Net Negative", yaxis_title="Amplification", height=600)
save_plotly(fig1, figout("01_topic_bubble_netneg_x_amp.html"))
fig1.show()

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952\figs\01_topic_bubble_netneg_x_amp.html


In [19]:
# Cell 12.2 — Lift Bar Chart (Top 15 SHEIN Exclusive Topics)
# -------------------------------------------------------------------
TOPK = 15
top_lift = viz.sort_values("lift_sheinLabor_vs_nonSheinLabor", ascending=False).head(TOPK).copy()

fig2 = px.bar(
    top_lift.iloc[::-1], 
    x="lift_sheinLabor_vs_nonSheinLabor", 
    y="label", 
    orientation="h", 
    color="net_negative",                 
    color_continuous_scale="RdBu_r", # 🌟 视觉修复：启用红蓝反向色阶 (Blue->Red)
    color_continuous_midpoint=0,     # 🌟 核心修复：强制 0 为中轴。红色=危险黑料，蓝色=正面品牌壁垒！
    hover_name="summary",                 
    hover_data={"examples": True, "topic_id": False, "n_comments": True},
    title=f"Top {TOPK} Brand-Specific Topics (Highly exclusive to SHEIN)"
)

fig2.update_layout(
    xaxis_title="Over-indexing Margin vs. Competitors (The larger, the more exclusive to us)", 
    yaxis_title="", 
    height=600,
    coloraxis_colorbar=dict(title="Sentiment<br>(Red=Risk, Blue=Win)") # 改写图例说明
)
save_plotly(fig2, figout("02_topic_lift_bar_primary.html"))
fig2.show()

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952\figs\02_topic_lift_bar_primary.html


In [20]:
# Cell 12.3 (REPLACE) — Priority Ranking (stance-aware destructiveness)
# -------------------------------------------------------------------
import plotly.express as px

viz = topic_lift[topic_lift["topic_id"] != -1].copy()
TOP_N = 25
top_p = viz.sort_values("priority", ascending=False).head(TOP_N).copy()

fig3 = px.bar(
    top_p.iloc[::-1], 
    x="priority",   
    y="label",       
    orientation="h", 
    color="lift_sheinLabor_vs_nonSheinLabor",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    hover_name="summary",
    hover_data={
        "topic_id": True,
        "n_comments": True,
        "priority": ':.2f',
        "destructiveness_base": ':.2f',
        "destructiveness_stance": ':.2f',
        "net_negative": ':.2f',
        "amplification": ':.1f',
        "resonance_wavg": ':.2f',
        "controversy_wavg": ':.2f',
        "avg_reply_weight": ':.1f',
        "threads_with_replies": True,
        "lift_sheinLabor_vs_nonSheinLabor": ':.3f',
    },
    title="Priority Ranking (zscore + NLI stance-aware) (Color=Lift)"
)

fig3.update_layout(xaxis_title="Priority (stance-aware destructiveness)", yaxis_title="Topic", height=780)
save_plotly(fig3, figout("03_priority_ranking_stance_aware.html"))
fig3.show()


✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952\figs\03_priority_ranking_stance_aware.html


In [30]:
# Cell 12.4 — Attribution Risk Matrix (Lift × Priority) [UPDATE HOVER]
# -------------------------------------------------------------------
import plotly.express as px
import numpy as np

viz2 = topic_lift[topic_lift["topic_id"] != -1].copy()

viz2["priority_log"] = np.log1p(viz2["priority"].fillna(0).clip(lower=0))
viz2["size_heat"] = np.log1p(viz2["avg_reply_weight"].fillna(0).clip(lower=0))

q05, q95 = viz2["controversy_wavg"].quantile([0.05, 0.95]).tolist()
if np.isclose(q05, q95):
    mid = float(viz2["controversy_wavg"].median())
    q05, q95 = mid - 0.05, mid + 0.05

viz2["topic_label"] = viz2["topic_id"].astype(str) + " - [" + viz2["keywords_short"].fillna("") + "]"

fig4 = px.scatter(
    viz2,
    x="lift_sheinLabor_vs_nonSheinLabor",
    y="priority_log",
    size="size_heat",
    color="controversy_wavg",
    range_color=[q05, q95],
    color_continuous_scale="Viridis",
    size_max=70,
    hover_name="topic_label",
    hover_data={
        "summary": True,
        "n_comments": True,
        "lift_sheinLabor_vs_nonSheinLabor": ':.3f',
        "priority": ':.2f',
        "destructiveness_stance": ':.2f',
        "net_negative": ':.2f',
        "amplification": ':.1f',
        # ✅ new resonance definition
        "resonance_strength_wavg": ':.2f',
        "polarity_wavg": ':.2f',
        "alignment_wavg": ':.2f',
        # keep controversy + heat
        "controversy_wavg": ':.2f',
        "avg_reply_weight": ':.1f',
        "threads_with_replies": True,
        "priority_log": False,
        "size_heat": False,
        "topic_label": False
    },
    title="Attribution Risk Matrix: Lift × log1p(Priority) (Size=log1p(Heat), Color=Controversy[5%-95%])"
)

fig4.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig4.update_layout(
    xaxis_title="Lift (SHEIN Labor - Non-SHEIN Labor)",
    yaxis_title="log1p(Priority)",
    height=650
)

save_plotly(fig4, figout("04_attribution_risk_matrix.html"))
fig4.show()


✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952\figs\04_attribution_risk_matrix.html


In [22]:
# Cell 13 — Export all assets
# -------------------------------------------------------------------
import os
from datetime import datetime, timezone
import json

# 🌟 核心修复点：把惨遭覆盖的 out 函数重新定义回来！
def out(filename):
    return os.path.join(RUN_DIR, filename)

OUT_COMMENT_TOPICS = out("comment_topics.csv")
OUT_TOPIC_SUMMARY = out("topic_summary.csv")
OUT_TOPIC_LIFT = out("topic_contrast_lift.csv")

# 1. 导出主楼带有 topic 标签的数据
df.to_csv(OUT_COMMENT_TOPICS, index=False, encoding="utf-8")

# 2. 导出话题核心指标、高质量标签和摘要
topic_summary.to_csv(OUT_TOPIC_SUMMARY, index=False, encoding="utf-8")

# 3. 导出交叉对比 Lift (含专属超标率)
topic_lift.to_csv(OUT_TOPIC_LIFT, index=False, encoding="utf-8")

# 4. 导出元数据报告
report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "n_comments_analyzed": len(df),
    "n_topics_found": len(topic_summary),
    "model_embedding": "all-MiniLM-L6-v2",
    "model_summarizer": "distilbart-cnn-12-6"
}
with open(out("report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("✅ 所有资产已成功导出！")
print("📁 文件夹路径:", RUN_DIR)

✅ 所有资产已成功导出！
📁 文件夹路径: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_08_topic_modeling_20260301_045952
